# Musanga V4 — NBAA: Annotation Consistency — exploratory audit

**Purpose:** explore *two distinct annotation tasks* before modelling: (1) five experts' `Musanga / non-Musanga / uncertain` classifications on the same candidate crowns, and (2) independently digitized Musanga-positive crown polygons in six verification circles by **four** annotators. Polygon absence is **not** automatically a non-Musanga vote.

**Scientific safeguards:** exact `verify_id` joins; no ground-truth claims or species inference; no models trained; no changes to V3_1/V4 reference or folds; spatial coordinates reprojected to UTM 35N before area/overlap calculations; independent polygon sets are *not* matched by `crown_id` (local per-annotator IDs). Comparisons of polygon overlap are restricted to circles both experts annotated. Shape uncertainty and annotation completeness are flagged, not silently resolved.

Read-only source: `Final/Data/Annotations/Verification/`. All outputs: `Final/Outputs/v4/NBAA_annotation_consistency/`.

**Run cells sequentially in Colab.** Large mosaics and LAS files are not accessed. This notebook is for initial scientific data reconnaissance, not annotator-specific classification training.

## 00 — V3_1-compatible Drive setup and file inventory

In [ ]:
%pip -q install geopandas shapely pyogrio scipy scikit-learn pandas matplotlib
from pathlib import Path
import json, re, math, warnings, itertools
from collections import Counter, defaultdict
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import cohen_kappa_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None,'Original Musanga Final directory unavailable'
DATA=ROOT/'Data'
SRC=DATA/'Annotations'/'Verification'
assert SRC.is_dir(),f'Expected input directory missing: {SRC}'
OUT=ROOT/'Outputs/v4/NBAA_annotation_consistency'
OUT.mkdir(parents=True,exist_ok=True)
files=sorted(p for p in SRC.iterdir() if p.is_file())
inventory=pd.DataFrame([dict(filename=p.name,extension=p.suffix.lower() or '(none)',
                             size_bytes=p.stat().st_size) for p in files])
inventory.to_csv(OUT/'NBAA_00_file_inventory.csv',index=False)
display(inventory)
print('Read-only source:',SRC)
print('Write-only NBAA output:',OUT)
print('Reference V3_1 / V4 datasets are NOT edited.')

## 01 — Classification CSV audit: exact crown IDs, labels and missingness

In [ ]:
CLASS_CSV=sorted(SRC.glob('musanga_classifications_*.csv'))
assert len(CLASS_CSV)==5, f'Expected 5 independent CSVs, found {[p.name for p in CLASS_CSV]}'
def who(p):
    n=p.name.casefold()
    if 'nadine' in n:return 'Nadine'
    if 'marjin' in n or 'marijn' in n:return 'Marijn'
    if 'joseph' in n:return 'Joseph'
    if 'thomas' in n:return 'Thomas'
    if 'kristof' in n:return 'Kristof'
    raise ValueError('Unknown annotator '+str(p))
NORM={'musanga':'M','non_musanga':'N','non-musanga':'N','non musanga':'N','uncertain':'U',
      'm':'M','n':'N','u':'U'}
raw_tables={};qa_rows=[];table_rows=[]
for p in CLASS_CSV:
    person=who(p);d=pd.read_csv(p,dtype={'verify_id':str})
    required={'verify_id','circle_id','classification'}
    assert required.issubset(d.columns),(p.name,required-set(d.columns))
    d['verify_id']=d.verify_id.astype(str).str.strip()
    d['label']=d.classification.astype(str).str.strip().str.casefold().map(NORM)
    qa_rows.append(dict(annotator=person,file=p.name,n_rows=len(d),
        unique_ids=d.verify_id.nunique(),duplicate_ids=int(d.verify_id.duplicated().sum()),
        invalid_labels=int(d.label.isna().sum()),circles=','.join(map(str,sorted(d.circle_id.dropna().unique()))),
        complete_flag_values=str(sorted(d.annotation_complete.dropna().astype(str).unique().tolist()))
             if 'annotation_complete' in d else '(absent)'))
    if d.verify_id.duplicated().any():raise ValueError(f'Duplicate verify_id in {p.name}')
    if d.label.isna().any():raise ValueError(f'Unexpected classification labels in {p.name}')
    raw_tables[person]=d
    table_rows.append(d[['verify_id','circle_id','label']].rename(columns={'circle_id':f'circle_{person}',
                                                                          'label':person}))
QA=pd.DataFrame(qa_rows).sort_values('annotator')
display(QA)
QA.to_csv(OUT/'NBAA_01_classification_file_audit.csv',index=False)
sets={who(p):set(raw_tables[who(p)].verify_id) for p in CLASS_CSV}
all_ids=set.union(*sets.values());common=set.intersection(*sets.values())
print('Union of IDs:',len(all_ids),'IDs present for every annotator:',len(common))
if len(common)!=len(all_ids):
    print('Missing IDs by annotator:',{name:sorted(all_ids-ids) for name,ids in sets.items()})
    print('Only complete matched IDs will enter five-expert agreement summaries.')
all_table=pd.DataFrame({'verify_id':sorted(all_ids)})
for t in table_rows:all_table=all_table.merge(t,on='verify_id',how='left',validate='one_to_one')
for person in raw_tables:
    derived=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
    bad=all_table[f'circle_{person}'].notna()&(derived!=all_table[f'circle_{person}'].astype('Int64'))
    assert not bad.any(),f'circle_id vs verify_id disagreement for {person}'
all_table['circle_id']=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
EXPERTS=['Nadine','Thomas','Joseph','Marijn','Kristof']
assert set(EXPERTS)==set(raw_tables)
matched=all_table.dropna(subset=EXPERTS).copy()
matched[['verify_id','circle_id',*EXPERTS]].to_csv(OUT/'NBAA_01_aligned_labels.csv',index=False)
display(matched.groupby('circle_id').size().rename('candidate_crowns').reset_index())
display(pd.DataFrame({person:matched[person].value_counts() for person in EXPERTS}).fillna(0).astype(int))
print('No label harmonization beyond the explicit M/N/U code mapping.')

## 02 — Classification consistency (descriptive, not annotator 'accuracy')

In [ ]:
stats=[]
for a,b in itertools.combinations(EXPERTS,2):
    q=matched[[a,b]].dropna();definite=q[(q[a]!='U')&(q[b]!='U')]
    stats.append(dict(annotator_a=a,annotator_b=b,n=len(q),
        agreement=float((q[a]==q[b]).mean()),
        cohen_kappa=float(cohen_kappa_score(q[a],q[b],labels=['M','N','U'])),
        n_definite=len(definite),
        definite_agreement=float((definite[a]==definite[b]).mean()) if len(definite) else np.nan,
        definite_kappa=float(cohen_kappa_score(definite[a],definite[b],labels=['M','N']))
           if len(definite) and definite[a].nunique()>1 and definite[b].nunique()>1 else np.nan))
PAIR=pd.DataFrame(stats)
PAIR.to_csv(OUT/'NBAA_02_pairwise_classifications.csv',index=False)
display(PAIR.round(3))
votes=matched[EXPERTS].apply(lambda r:r.value_counts(),axis=1).fillna(0).astype(int)
for k in ['M','N','U']:
    matched[f'votes_{k}']=votes[k].values if k in votes else 0
matched['largest_agreement']=matched[[f'votes_{x}' for x in 'MNU']].max(axis=1)
matched['majority_label']=matched.apply(lambda r: next((k for k in 'MNU' if r[f'votes_{k}']>=3),'none'),axis=1)
display(matched.groupby(['circle_id','largest_agreement']).size().unstack(fill_value=0))
display(matched.groupby('majority_label').size().rename('crowns'))
matched.to_csv(OUT/'NBAA_02_crown_vote_table.csv',index=False)
print('NOTE: agreement measures label similarity, NOT botanical species correctness.')

## 03 — Polygon geometry audit and in-memory topology repair

Some digitized polygons are self-intersecting or topologically invalid. Preserve original geometry quality in a CSV and use `make_valid` on an in-memory analytical copy before unions and IoU. Geometry changes are quantified; source files are never modified.

In [ ]:
POLY_FILES=sorted(p for p in SRC.iterdir() if p.name.startswith('Musanga_Polygones_'))
print('Polygon export files:',[p.name for p in POLY_FILES])
assert len(POLY_FILES)==4, f'Expected 4 polygon exports; found {len(POLY_FILES)}'
from shapely import make_valid
from shapely.geometry import MultiPolygon, Polygon, GeometryCollection
from shapely.ops import unary_union

def polygonal_parts(geom):
    """Preserve only actual polygon components after GEOS repair."""
    if geom is None or geom.is_empty:return []
    if isinstance(geom,Polygon):return [geom]
    if isinstance(geom,MultiPolygon):return list(geom.geoms)
    if hasattr(geom,'geoms'):
        return [p for g in geom.geoms for p in polygonal_parts(g)]
    return []

def repair_polygon_for_analysis(geom):
    """Only an in-memory analysis copy; never overwrite or silently fix source."""
    if geom is None or geom.is_empty:return None
    fixed=make_valid(geom) if not geom.is_valid else geom
    polys=[p for p in polygonal_parts(fixed) if p.is_valid and p.area>0]
    if not polys:return None
    result=unary_union(polys)
    return result if not result.is_empty else None

def safe_union(geometries):
    cleaned=[g for g in geometries if g is not None and not g.is_empty]
    if not cleaned:return GeometryCollection()
    return unary_union(cleaned)

POLY={}
poly_quality=[]
GEOM_QA=[]
UTM='EPSG:32635'
for p in POLY_FILES:
    person=who(p)
    raw=json.loads(p.read_text(encoding='utf-8-sig'))
    assert raw.get('type')=='FeatureCollection' and isinstance(raw.get('features'),list),p.name
    entries=[]
    for i,f in enumerate(raw['features']):
        props=f.get('properties') or {}
        geom=shape(f['geometry']) if f.get('geometry') else None
        entries.append({**props,'geometry':geom,'source_row':i})
    orig=gpd.GeoDataFrame(entries,geometry='geometry',crs='EPSG:4326')
    assert 'circle_id' in orig and 'crown_id' in orig,p.name
    orig['circle_id']=pd.to_numeric(orig.circle_id,errors='raise').astype(int)
    # Work in metric CRS. Keep the raw geometry independently for audit.
    orig=orig.to_crs(UTM)
    clean_geoms=[]
    for _,r in orig.iterrows():
        geom=r.geometry
        was_valid=(geom is not None and not geom.is_empty and geom.is_valid)
        original_area=float(geom.area) if geom is not None and not geom.is_empty else 0.
        fixed=repair_polygon_for_analysis(geom)
        repaired_area=float(fixed.area) if fixed is not None else np.nan
        GEOM_QA.append(dict(annotator=person,source_row=int(r.source_row),
             circle_id=int(r.circle_id),crown_id=str(r.crown_id),
             original_valid=bool(was_valid),original_geom_type=geom.geom_type if geom is not None else '',
             original_area_m2=original_area,repaired_area_m2=repaired_area,
             area_change_pct=100*(repaired_area-original_area)/original_area if original_area>0 else np.nan,
             geometry_usable=bool(fixed is not None),
             repaired=bool(not was_valid and fixed is not None)))
        clean_geoms.append(fixed)
    # All downstream area/IoU/matching/visualization operations use validated copies.
    d=orig.copy()
    d['geometry']=gpd.GeoSeries(clean_geoms,index=d.index,crs=UTM)
    if d.geometry.isna().any():
        print('WARNING: unusable polygon geometry excluded from spatial comparisons:',
              person,d.loc[d.geometry.isna(),['circle_id','crown_id','source_row']].to_dict('records'))
        d=d.loc[d.geometry.notna()].copy()
    assert d.geometry.is_valid.all(),f'Invalid geometry remains for {person}'
    d['area_m2']=d.geometry.area
    d['centroid_x']=d.geometry.centroid.x
    d['centroid_y']=d.geometry.centroid.y
    d['annotator']=person
    POLY[person]=d
    for cid,g in d.groupby('circle_id'):
        geom_union=safe_union(g.geometry.tolist())
        rows_for_circle=[x for x in GEOM_QA if x['annotator']==person and x['circle_id']==int(cid)]
        poly_quality.append(dict(annotator=person,circle_id=int(cid),n_polygons=len(g),
             sum_polygon_area_m2=float(g.area_m2.sum()),
             union_area_m2=float(geom_union.area),
             median_polygon_area_m2=float(g.area_m2.median()),
             duplicate_crown_ids=int(g.crown_id.duplicated().sum()),
             n_invalid_original=sum(not x['original_valid'] for x in rows_for_circle),
             n_repaired=sum(x['repaired'] for x in rows_for_circle),
             n_unusable=sum(not x['geometry_usable'] for x in rows_for_circle),
             annotation_status=str(g.annotation_status.value_counts().to_dict())
                 if 'annotation_status' in g else '(absent)'))
GEOM_AUDIT=pd.DataFrame(GEOM_QA)
GEOM_AUDIT.to_csv(OUT/'NBAA_03_geometry_repair_audit.csv',index=False)
POLY_QA=pd.DataFrame(poly_quality).sort_values(['circle_id','annotator'])
POLY_QA.to_csv(OUT/'NBAA_03_polygon_coverage_by_circle.csv',index=False)
display(POLY_QA.round(2))
print('=== Original geometry validity and in-memory repairs ===')
display(GEOM_AUDIT.groupby('annotator').agg(
    original_invalid=('original_valid',lambda x:int((~x).sum())),
    repaired=('repaired','sum'),unusable=('geometry_usable',lambda x:int((~x).sum())),
    max_abs_area_change_pct=('area_change_pct',lambda x:float(x.abs().max()))).round(3))
matrix=POLY_QA.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int)
display(matrix)
print('Usable polygon counts:',{a:len(d) for a,d in POLY.items()})
print('All original source files are unchanged. Subsequent sections use repaired analytical copies.')
print('Crucial: missing polygon != non-Musanga classification; completeness must be established.')


## 04 — Pairwise *spatial* agreement where both annotators drew polygons

In [ ]:
# Area-level IoU on comparable circle-specific polygon unions.
# Detection overlap alone cannot distinguish two species without independent truth.
pair_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    overlap_circles=sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id))
    for cid in overlap_circles:
        aa=POLY[a].loc[POLY[a].circle_id==cid]
        bb=POLY[b].loc[POLY[b].circle_id==cid]
        ga=unary_union(aa.geometry.tolist());gb=unary_union(bb.geometry.tolist())
        inter=ga.intersection(gb).area;uni=ga.union(gb).area
        pair_rows.append(dict(circle_id=cid,annotator_a=a,annotator_b=b,
          n_a=len(aa),n_b=len(bb),area_a_m2=float(ga.area),area_b_m2=float(gb.area),
          intersection_m2=float(inter),union_m2=float(uni),
          area_iou=float(inter/uni) if uni>0 else np.nan,
          a_covered_by_b=float(inter/ga.area) if ga.area>0 else np.nan,
          b_covered_by_a=float(inter/gb.area) if gb.area>0 else np.nan))
SPATIAL=pd.DataFrame(pair_rows)
SPATIAL.to_csv(OUT/'NBAA_04_pairwise_polygon_union_IoU.csv',index=False)
display(SPATIAL.sort_values(['circle_id','annotator_a','annotator_b']).round(3))
display(SPATIAL.groupby(['annotator_a','annotator_b']).agg(
  circles=('circle_id','count'),mean_area_IoU=('area_iou','mean'),
  median_area_IoU=('area_iou','median')).round(3))
print('These are unsigned overlap measures; polygons can represent different crowns or delineation rules.')

## 05 — Crown-to-crown spatial matching (exploratory, threshold sensitivity)

In [ ]:
def match_iou(A,B,minimum):
    # IoU is computed in projected metres; optimal one-to-one Hungarian assignment.
    if not len(A) or not len(B):return dict(matches=0,iou_mean=np.nan,frac_a=0,frac_b=0)
    mat=np.zeros((len(A),len(B)),dtype=float)
    for i,ga in enumerate(A.geometry):
        for j,gb in enumerate(B.geometry):
            if not ga.intersects(gb):continue
            u=ga.union(gb).area
            mat[i,j]=ga.intersection(gb).area/u if u>0 else 0
    rr,cc=linear_sum_assignment(-mat)
    good=mat[rr,cc]>=minimum
    return dict(matches=int(good.sum()),
        iou_mean=float(mat[rr[good],cc[good]].mean()) if good.any() else np.nan,
        frac_a=float(good.sum()/len(A)),frac_b=float(good.sum()/len(B)))
match_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    for cid in sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id)):
        da=POLY[a].loc[POLY[a].circle_id==cid]
        db=POLY[b].loc[POLY[b].circle_id==cid]
        for threshold in (.10,.30,.50):
            match_rows.append(dict(circle_id=cid,a=a,b=b,iou_threshold=threshold,
                n_a=len(da),n_b=len(db),**match_iou(da,db,threshold)))
MATCH=pd.DataFrame(match_rows)
MATCH.to_csv(OUT/'NBAA_05_one_to_one_crown_matches.csv',index=False)
display(MATCH.groupby(['a','b','iou_threshold']).agg(
    circles=('circle_id','count'),matched_crowns=('matches','sum'),
    avg_fraction_a=('frac_a','mean'),avg_fraction_b=('frac_b','mean')).round(3))
print('Unmatched crown need not be a false positive: reference completeness is unverified.')

## 06 — Compact site map / visual QA and exploration hand-off

In [ ]:
# Small vector-only visualization: no large TIFF reads.
from matplotlib.patches import Patch
colors={'Nadine':'#1e7962','Thomas':'#cf7c37','Joseph':'#6178b8','Marijn':'#ad589c'}
circles=sorted(set.union(*(set(d.circle_id) for d in POLY.values())))
fig,axes=plt.subplots(2,3,figsize=(15,10))
for ax,cid in zip(axes.flat,circles):
    for person in ['Nadine','Thomas','Joseph','Marijn']:
        d=POLY.get(person)
        if d is None:continue
        subset=d[d.circle_id==cid]
        if len(subset):subset.boundary.plot(ax=ax,color=colors[person],linewidth=1,alpha=.8)
    ax.set_title(f'Circle {cid} — independently digitized crowns')
    ax.set_aspect('equal');ax.tick_params(labelsize=7)
for ax in axes.flat[len(circles):]:ax.axis('off')
fig.legend(handles=[Patch(facecolor='none',edgecolor=colors[n],label=n) for n in colors],
           loc='lower center',ncol=4)
plt.tight_layout(rect=[0,.04,1,1])
fig.savefig(OUT/'NBAA_06_polygon_overlay_by_circle.png',dpi=170)
plt.show();plt.close(fig)
contract=dict(schema='NBAA_v4_exploratory_1',
  source_directory=str(SRC),output_directory=str(OUT),
  classification_files={p:raw_tables[p].shape[0] for p in EXPERTS},
  common_classification_crowns=int(len(matched)),
  digitized_polygon_counts={p:int(len(POLY[p])) for p in POLY},
  classification_policy='Uncertain is a separate observed label; missing is not negative',
  polygon_policy='Independent positive polygons; absent polygons not negative; no automatic truth',
  alignment='CSV verify_id joins; polygon overlays by projected geometry and circle',
  train_test_policy='No training or new splits in this audit',
  further_review=['Digitizing completeness for each circle and annotator',
                  'Compare classification vs polygon tasks only where identity/coverage established',
                  'Independent RGB/DSM embeddings and spatially held-out annotator predictors',
                  'Repeat blinded annotations to assess genuine intra-annotator repeatability'])
(OUT/'NBAA_manifest.json').write_text(json.dumps(contract,indent=2))
print('NBAA exploration complete. Outputs:')
for p in sorted(OUT.glob('NBAA_*')):print(' ',p.name)
print('NEXT DECISION: review counts, completeness, overlap and image QA before modelling.')

## 07 — Which annotators outlined the **same crown**? (exploratory spatial object agreement)

**Critical distinction:** a zero in the polygon count table means **no polygon record**, *not* an observed zero-Musanga survey. Every circle without polygons is marked `not_recorded`; circles with polygons are marked `polygons_present_completion_unverified`, because all source polygon records have `annotation_status='draft'`. Only independently confirmed complete surveys can support false-negative/detection-rate interpretation.

This block identifies spatially overlapping polygon groups. An edge requires **different annotators**, geometrical intersection, polygon IoU ≥0.10 *or* at least 35% of the smaller polygon covered, plus nearby centroids. Connected groups with **multiple polygons from the same annotator** are flagged `ambiguous_group` and *not included in reliable 1/2/3/4 agreement counts*. Results are sensitive to these explicitly shown thresholds. Spatial overlap is not a species-truth vote.

Outputs: per-circle coverage, per-crown member table, group-level number of annotators (1–4), counts by circle, diagnostics/ambiguity, and plots. Visual overlay displays are vector-based; a georeferenced UAV RGB clip is added only if found locally and raster windows can be read safely.

In [ ]:
# NBAA Section 07A — coverage matrix and conservative spatial crown-group graph.
# Only reuses already-loaded POLY (repaired, independent) and reads no mosaics.
from collections import defaultdict
from itertools import combinations
from scipy.spatial.distance import euclidean
from shapely.geometry import box
import networkx as nx
try:
    import rasterio
    from rasterio.windows import from_bounds as raster_window_from_bounds
except ImportError:
    rasterio=None

annotators=['Nadine','Thomas','Joseph','Marijn']
circles=sorted(set().union(*[set(df.circle_id.unique()) for df in POLY.values()]))
coverage=[]
for circle in circles:
    for person in annotators:
        df=POLY[person];part=df[df.circle_id==circle]
        coverage.append(dict(circle_id=int(circle),annotator=person,
          n_polygons=int(len(part)),
          status='polygons_present_completion_unverified' if len(part) else 'not_recorded',
          complete_verified=False))
COVERAGE=pd.DataFrame(coverage)
COVERAGE.to_csv(OUT/'NBAA_07_circle_annotator_coverage.csv',index=False)
display(COVERAGE.pivot(index='circle_id',columns='annotator',values='status'))
display(COVERAGE.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int))
print('CAUTION: zero records are unassessed, not confirmed zero Musanga.')
print('Even where polygons exist, full-circle completeness is not independently established.')

MIN_IOU=.10
MIN_SMALLER_OVERLAP=.35
MAX_CENTROID_FACTOR=1.5 # times the sum of equivalent-area circle radii
members=[];groups=[];edges=[]
for cid in circles:
    records=[]
    for who_ in annotators:
        d=POLY[who_]
        for _,r in d[d.circle_id==cid].iterrows():
            records.append(dict(annotator=who_,crown_id=str(r.crown_id),
                source_row=int(r.source_row),geometry=r.geometry,
                area_m2=float(r.geometry.area)))
    graph=nx.Graph();graph.add_nodes_from(range(len(records)))
    for i,j in combinations(range(len(records)),2):
        a,b=records[i],records[j]
        if a['annotator']==b['annotator']:continue
        ga,gb=a['geometry'],b['geometry']
        if not ga.intersects(gb):continue
        inter=ga.intersection(gb).area
        if inter<=0:continue
        union=ga.area+gb.area-inter
        iou=inter/union if union else 0
        smaller=inter/min(ga.area,gb.area) if min(ga.area,gb.area)>0 else 0
        ra=np.sqrt(ga.area/np.pi);rb=np.sqrt(gb.area/np.pi)
        dist=ga.centroid.distance(gb.centroid)
        if (iou>=MIN_IOU or smaller>=MIN_SMALLER_OVERLAP) and dist<=MAX_CENTROID_FACTOR*(ra+rb):
            graph.add_edge(i,j,weight=float(iou))
            edges.append(dict(circle_id=int(cid),annotator_a=a['annotator'],annotator_b=b['annotator'],
              crown_a=a['crown_id'],crown_b=b['crown_id'],
              iou=float(iou),smaller_overlap=float(smaller),centroid_distance_m=float(dist)))
    for group_idx,comp in enumerate(sorted(nx.connected_components(graph),key=lambda s:min(s)),start=1):
        rows=[records[i] for i in sorted(comp)]
        unique_people={r['annotator'] for r in rows}
        collisions=[person for person in unique_people if sum(r['annotator']==person for r in rows)>1]
        ambiguous=bool(collisions)
        gid=f'C{cid:02d}_G{group_idx:03d}'
        geoms=[r['geometry'] for r in rows]
        union=safe_union(geoms)
        groups.append(dict(group_id=gid,circle_id=int(cid),
            n_polygons=len(rows),n_annotators=len(unique_people),
            annotators=';'.join(sorted(unique_people)),
            ambiguous_group=ambiguous,
            same_annotator_collisions=';'.join(sorted(collisions)),
            member_ids=';'.join(f"{r['annotator']}:{r['crown_id']}" for r in rows),
            centroid_easting=float(union.centroid.x),
            centroid_northing=float(union.centroid.y),
            union_area_m2=float(union.area)))
        for r in rows:
            members.append(dict(group_id=gid,circle_id=int(cid),
                annotator=r['annotator'],crown_id=r['crown_id'],
                source_row=r['source_row'],polygon_area_m2=r['area_m2'],
                ambiguous_group=ambiguous))
GROUPS=pd.DataFrame(groups)
MEMBERS=pd.DataFrame(members)
EDGES=pd.DataFrame(edges)
for name,frame in [('NBAA_07_crown_groups.csv',GROUPS),
                    ('NBAA_07_crown_members.csv',MEMBERS),
                    ('NBAA_07_spatial_match_edges.csv',EDGES)]:
    frame.to_csv(OUT/name,index=False)
print('Spatial crown groups (not verified true crowns):',len(GROUPS))
print('Ambiguous chained groups:',int(GROUPS.ambiguous_group.sum()))
display(GROUPS.groupby(['n_annotators','ambiguous_group']).size().rename('groups').reset_index())
reliable=GROUPS.loc[~GROUPS.ambiguous_group]
print('Crown groups by number of annotators, excluding ambiguous links:')
display(reliable.groupby(['circle_id','n_annotators']).size().unstack(fill_value=0).reindex(columns=[1,2,3,4],fill_value=0))
print('Full group identity table (inspect before interpreting consensus):')
display(GROUPS[['group_id','circle_id','n_polygons','n_annotators','annotators',
                'ambiguous_group','member_ids']].head(35))


## 08 — Visual check: individual crown groups and six-circle overlay

The following vector overlays show whether purported matches correspond to the **same crown** or are merely touching boundaries. If georeferenced `circle_XX_UAV_RGB.tif` clips from `06_GEE_Verification_v2` are available under the existing `Final/Outputs` tree, they are shown as backgrounds using small downsampled raster windows. **Do not trust automated group counts until reviewing the overlays**, especially zero-intersection cases and flagged ambiguous groups.

In [ ]:
# NBAA Section 08 — small per-circle visual audit with per-group numbers.
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
COLORS={'Nadine':'#158466','Thomas':'#e09025','Joseph':'#3b70c1','Marijn':'#ae50a1'}
# Locate old V3_1 GEE verification circle clips, if already mounted; do not download giant mosaics.
paths={int(p.stem.split('_')[1]):p for p in ROOT.rglob('circle_??_UAV_RGB.tif')
       if '06_GEE_Verification' in str(p) and p.stem.split('_')[1].isdigit()}
print('Previously prepared UAV circle clips found:',{k:str(v) for k,v in paths.items()})
fig,axes=plt.subplots(2,3,figsize=(17,11))
for ax,cid in zip(axes.flat,circles):
    # source polygons are already projected UTM 35N
    parts={a:POLY[a].loc[POLY[a].circle_id==cid] for a in annotators}
    all_geoms=[g for d in parts.values() for g in d.geometry]
    if not all_geoms:ax.axis('off');continue
    bounds=safe_union(all_geoms).bounds
    pad=12
    extent=(bounds[0]-pad,bounds[2]+pad,bounds[1]-pad,bounds[3]+pad)
    image_used=False
    if rasterio is not None and cid in paths:
        try:
            with rasterio.open(paths[cid]) as ds:
                if str(ds.crs).upper()=='EPSG:32635':
                    win=raster_window_from_bounds(extent[0],extent[2],extent[1],extent[3],transform=ds.transform)
                    import rasterio.windows
                    win=win.intersection(rasterio.windows.Window(0,0,ds.width,ds.height))
                    if win.width>1 and win.height>1:
                        arr=ds.read([1,2,3],window=win,out_shape=(3,700,700),
                                    resampling=rasterio.enums.Resampling.bilinear)
                        bb=rasterio.windows.bounds(win,ds.transform)
                        ax.imshow(np.moveaxis(arr,0,-1),extent=[bb[0],bb[2],bb[1],bb[3]])
                        image_used=True
        except Exception as err: print(f'RGB background unavailable for {cid}:',str(err)[:110])
    for who_ in annotators:
        p=parts[who_]
        if len(p):p.boundary.plot(ax=ax,edgecolor=COLORS[who_],linewidth=1.8,alpha=.9)
    for _,r in GROUPS[GROUPS.circle_id==cid].iterrows():
        ax.text(r.centroid_easting,r.centroid_northing,
             r.group_id.split('_')[-1].lstrip('G')+('!' if r.ambiguous_group else ''),
             fontsize=7,color='white' if image_used else 'black',
             ha='center',va='center',
             bbox=dict(facecolor='black' if image_used else 'white',alpha=.5,
                       edgecolor='none',pad=.3))
    ax.set(xlim=extent[:2],ylim=extent[2:],title=f'Circle {cid} (RGB: {"yes" if image_used else "not found"})')
    ax.set_aspect('equal');ax.tick_params(labelsize=7)
for ax in axes.flat[len(circles):]:ax.axis('off')
fig.legend(handles=[Patch(facecolor='none',edgecolor=COLORS[a],label=a) for a in annotators],
    loc='lower center',ncol=4)
fig.suptitle('Spatial crown identity exploration — ! = ambiguous chain; missing surveys are NOT zeros')
plt.tight_layout(rect=[0,.045,1,.96])
fig.savefig(OUT/'NBAA_08_four_annotator_crown_group_overlay.png',dpi=180)
plt.show();plt.close(fig)
print('Review the overlapping polygons visually before interpreting n_annotators.')
print('The four digitized polygon files alone do NOT prove that all four surveyed each circle.')
